## Ejercicios de Integración: 

**Reglas de Simpson:** 

**a) Simpson Compuesta** 

Esta es una extensión de la regla de Simpson simple vista en clase, lo que diferencia a esta de la regla simple es que se aplica muchas veces sobre subintervalos pequeños, es decir, si tenemos un intervalo $[a,b]$ lo dividimos en "N" sub intervalos de paso:  $$ h = \frac{b-a}{N}$$ Bajo las condiciones de que N debe ser par, con forma general: $$ \frac{h}{3} [f(x_0) +4\sum_{i\ impares}f(x_i) + 2\sum_{i\ pares}f(x_i) + f(x_N)]$$

**b) Simpson 3/8** 

Esta es una variante de la regla de Simpson simple que en lugar de aproximar la función con una parábola usa un polinomio de grado que pasa por 4 puntos, su forma general es: $$\frac{3h}{8}[f(x_0) +3f(x_1) + 3f(x_2) +f(x_3)]$$

**Ejercicios:** 

**a) Integral Parabola**

Hallar el área exacta de la región entre la parábola $y=x^3$ y el eje $x$ en el intervalo $[0,b]$ usando las sumas de Riemann hacia la derecha de igual ancho. 
Para esto primero estableceremos $a=0$, así nuestro intervalo es $[0,b]$ y nuestro pasos son $\triangle x = \frac{b-0}{n}$, con esto definimos los puntos de evaluación como $x_i = i \triangle x$, con todo esto definimos las sumas de Riemann a derecha: $$ S_n = \sum_{i=1} f(x_i) \triangle x $$ Ahora, si tenemos en cuenta que nuestra función es $f(x_i)= (x_i)^3$ entonces tenemos: $$S_n = \sum_{i=1} (\frac{ib}{n})^3 \frac{b}{n} $$ $$ S_n = (\frac{b}{n})^4 \sum_{i=1} i^3$$ La cual ya es una suma conocida, así que usando el resultado tenemos: $$ S_n = (\frac{b}{n})^4 (\frac{n(n+1)}{2})^2$$ Luego, para sacar el área exácta evaluamos el límite de $S_n$ cuando n tiende a infinito tal que: $$A = \lim_{n \to \infty} S_n$$  $$ \lim_{n \to \infty} (\frac{b}{n})^4 (\frac{n(n+1)}{2})^2$$ $$\frac{b^4}{4} \lim_{n \to \infty}  \frac{(n+1)^2}{n^2} $$ $$ A = \frac{b^4}{4}$$ Que coincide con la integral exacta de la función $x^3$. 

**b) Cohete con múltiples métodos:**

Para este ejercicio nos piden resolver la integral de cohete con los métodos de trapezoide, trapezoide compuesto y Simpson 1/3, posteriormente hacer tabla de su comportamiento para diferentes n con el valor obtenido, error absoluto y error relativo. Para esto primero definimos las integrales utilizando los 3 métodos y posteriormente definirlas en código: 

Método de Simpson 1/3: $$S_n = \frac{\triangle x}{3}[f(a) + f(b) + \sum_{i\ impares} f(a+i\triangle x) \sum_{i\ pares} f(a + i\triangle x)]$$
Método del trapecio: $$ T_n = \frac{\triangle x}{2} (f(a) + 2\sum_{i=1}f(a+i\triangle x) + f(b))$$
Método de trapecio compuesto: $$T_C = \frac{h}{2} \left[ f(a) + 2 \sum_{i=1}^{n-1} f(x_i) + f(b) \right]$$ 


In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import quad

#DATOS 
ve = 2000.0    # Velocidad de expulsión [m/s]
q = 2100.0     # Consumo de combustible [kg/s]
m0 = 140000.0  # Masa inicial [kg]
g = 9.8        # Gravedad [m/s^2]
t1 = 8.0       # Tiempo inicial [s]
t2 = 30.0      # Tiempo final [s]
''
def f(t):
    return ve * np.log(m0 / (m0 - q * t)) - g * t

# Valor de referencia "exacto" para calcular errores (absoluto y relativo)
val_real, _ = quad(f, t1, t2)

#DEFINICIÓN DE MÉTODOS DE INTEGRACIÓN

def riemann_derecha(f, a, b, n):
    dx = (b - a) / n
    x = np.linspace(a, b, n + 1)
    return dx * np.sum(f(x[1:]))

def trapezoide_compuesto(f, a, b, n):
    """
    Cubre Trapezoide Simple (n=1) y Compuesto (n>1).
    Referencia: Imágenes 1 y 3.
    """
    dx = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    return (dx / 2) * (y[0] + 2 * np.sum(y[1:-1]) + y[-1])

def simpson_13_compuesto(f, a, b, n):
    """
    Regla de Simpson 1/3. Solo válida para n par.
    Referencia: Imagen 2.
    """
    if n % 2 != 0:
        return None  # Para identificar el N/A después
    
    dx = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    # Suma: y0 + 4*(sum impares) + 2*(sum pares) + yn
    suma = y[0] + 4 * np.sum(y[1:-1:2]) + 2 * np.sum(y[2:-1:2]) + y[-1]
    return (dx / 3) * suma

#BLOQUE 3: CÁLCULO DE N=1 A N=10
resultados = []

for n in range(1, 11):
    # Cálculos numéricos
    x_riemann = riemann_derecha(f, t1, t2, n)
    x_trap = trapezoide_compuesto(f, t1, t2, n)
    x_simp = simpson_13_compuesto(f, t1, t2, n)
    
    # Función auxiliar para calcular fila de errores, ESTA IDEA LA ROBÉ DE FELIPE Y SANTIAGO JAJA
    def obtener_metricas(val_calc):
        if val_calc is None:
            return "N/A", "N/A", "N/A"
        err_abs = abs(val_real - val_calc)
        err_rel = (err_abs / abs(val_real)) * 100
        return val_calc, err_abs, err_rel

    r_val, r_ea, r_er = obtener_metricas(x_riemann)
    t_val, t_ea, t_er = obtener_metricas(x_trap)
    s_val, s_ea, s_er = obtener_metricas(x_simp)
    
    resultados.append([
        n, r_val, r_ea, r_er, 
        t_val, t_ea, t_er, 
        s_val, s_ea, s_er
    ])

#BLOQUE 4: IMPRESIÓN DE LA TABLA
columnas = [
    'n', 'x Riemann', 'EA Riemann', 'ER Riemann %',
    'x Trap', 'EA Trap', 'ER Trap %',
    'x Simpson', 'EA Simpson', 'ER Simpson %'
]

df = pd.DataFrame(resultados, columns=columnas)

# Formateo para que se vea limpio (4 decimales para números)
pd.options.display.float_format = '{:.4f}'.format

print(f"VALOR REAL DE LA INTEGRAL: {val_real:.6f}\n")
print(df.to_string(index=False, justify='center'))

#Puede ser el sueño pero este código se me hizo super bola y tuve que dividirlo en bloques para no perderme, espero que se entienda bien. Cualquier duda me avisan! (por cierto, esa última parte la escribió copilot, que miedo los chat bots, Microsof tkm no divulgues mis datos) 

VALOR REAL DE LA INTEGRAL: 11061.335535

 n  x Riemann  EA Riemann  ER Riemann %   x Trap    EA Trap  ER Trap % x Simpson  EA Simpson ER Simpson %
 1 19836.8280  8775.4925     79.3348    11868.3482 807.0127   7.2958          N/A      N/A         N/A   
 2 15250.6142  4189.2787     37.8732    11266.3743 205.0388   1.8537   11065.7163   4.3808      0.0396   
 3 13808.9191  2747.5835     24.8395    11152.7591  91.4236   0.8265          N/A      N/A         N/A   
 4 13104.9406  2043.6051     18.4752    11112.8207  51.4851   0.4655   11061.6361   0.3006      0.0027   
 5 12687.9997  1626.6642     14.7059    11094.3038  32.9682   0.2980          N/A      N/A         N/A   
 6 12412.3168  1350.9813     12.2135    11084.2369  22.9013   0.2070   11061.3961   0.0606      0.0005   
 7 12216.5182  1155.1827     10.4434    11078.1640  16.8284   0.1521          N/A      N/A         N/A   
 8 12070.2813  1008.9457      9.1214    11074.2213  12.8858   0.1165   11061.3548   0.0193      0.0002   
 9 11

**c) Función que yo quiera con múltiples métodos:**

Para este ejercicio voy a elegir la función 